# recs_022 -- Chroma `game_profiles` / `game_review_chunks` EDA

QA pass on Stage 1/2's Chroma output before wiring `ChromaGameProfileRetriever` into eval (Stage 3).

# Executive Summary

**Result:** Both collections match the Stage 1/2 contract exactly -- counts, metadata,
unit-norm embeddings, and sensible nearest neighbors for two different games.

**Decision:** No data-layer blockers. Proceed to Stage 3.

# Business Context

Stage 3 is about to register a new eval method backed by these collections. Checking the
underlying data directly first, before trusting any Recall@K/Hit@K numbers built on it.

# Research Question

Do `game_review_chunks` and `game_profiles` match the Stage 1/2 contract -- counts, metadata,
embedding shape, sensible neighbors?

# Hypothesis

Counts: 16,010 chunk rows (315 games), 1,258 profile rows (315 x 4 variants, minus 2 skips).
Embeddings unit-norm. Half-Life retrieves genre-appropriate neighbors.

**Result: confirmed** -- see Key Findings.

# Definitions

| Term | Definition |
|------|------------|
| `chunk_type` | `"review"` or `"description"` |
| `variant` | `{any_polarity,recommended_only}__{flat,log_weighted}` |
| `blend_weight` | weight on the description vector when blending into `game_profiles` (default 0.1) |
| `n_reviews_pooled` | reviews feeding a given `(app_id, variant)` vector |
| distance | Chroma cosine distance, `1 - cosine_similarity`; lower = more similar |

# Data Sources

Both from `artifacts/recs/embeddings/game_chunks/chroma/`:

- **`game_review_chunks`** -- one row per review chunk (any-polarity top-50-by-`votes_helpful`,
  `min_review_chars>=30`) + one description row per game. Fine-grain, not itself queried for retrieval.
- **`game_profiles`** -- one row per `app_id x variant` (4 variants), blended pooled+description
  vector. This is what Stage 3 queries. 1 game skipped for both `recommended_only__*` variants
  (zero eligible reviews).

# Design / Process

Direct `chromadb` client inspection (counts, metadata) + `ChromaGameProfileRetriever.top_k()` for
neighbor spot checks. Data-layer QA, not a retrieval-quality ablation.

# Evaluation Outputs / Artifacts

| Artifact | Location | Generated by |
|---|---|---|
| `game_review_chunks` | `artifacts/recs/embeddings/game_chunks/chroma/` | `recs_job_game_chunk_embeddings.py` |
| `game_profiles` | `artifacts/recs/embeddings/game_chunks/chroma/` | `recs_job_game_chunk_embeddings.py` |

# Notebook Roadmap

1. Load collections
2. `game_review_chunks` counts + metadata
3. `game_profiles` counts + metadata (per variant)
4. Embedding norm check
5. Nearest-neighbor spot checks

# Analysis

## Setup

In [1]:
import json
from pathlib import Path

import chromadb
import numpy as np
import pandas as pd


def _find_repo_root(start: Path) -> Path:
    here = start.resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from start={start}")


REPO_ROOT = _find_repo_root(Path.cwd())
CHROMA_PERSIST_DIR = REPO_ROOT / "artifacts" / "recs" / "embeddings" / "game_chunks" / "chroma"

pd.options.display.max_columns = 20
pd.options.display.width = 140

client = chromadb.PersistentClient(path=str(CHROMA_PERSIST_DIR))
[c.name for c in client.list_collections()]

['game_review_chunks', 'game_profiles']

In [2]:
client.count_collections()

2

## Load Data

In [3]:
review_chunks = client.get_collection("game_review_chunks")
game_profiles = client.get_collection("game_profiles")

print("game_review_chunks count:", review_chunks.count())
print("game_profiles count:", game_profiles.count())

game_review_chunks count: 16010
game_profiles count: 1258


## Validate Data Quality — `game_review_chunks`

In [4]:
review_chunks.get(include=["metadatas", "documents", "embeddings"], limit=2)

{'ids': ['70_description', '70_2380658'],
 'embeddings': array([[-0.03604267,  0.0237489 , -0.01157143, ..., -0.05596663,
          0.05810434, -0.03119762],
        [-0.03290794,  0.00921143, -0.00040082, ..., -0.02256973,
          0.07829778, -0.00427932]], shape=(2, 512)),
 'documents': ['Half-Life is a 1998 first-person shooter (FPS) game developed by Valve Corporation and published by Sierra Studios for Windows. It was Valve\'s debut product and the first game in the Half-Life series. The player assumes the role of Gordon Freeman, a theoretical physicist who must escape from the Black Mesa Research Facility after it is overrun by aliens following a disastrous scientific experiment. Its gameplay consists of diverse combat, exploration and puzzles.\n\nDr. Gordon Freeman arrives late for work at 8:47 am in the Black Mesa Research Facility, using the advanced Black Mesa tram system that leads through the facility. He arrives at the Anomalous Materials Lab, his work place, and he is i

In [5]:
all_chunks = review_chunks.get(include=["metadatas"])
meta_df = pd.DataFrame(all_chunks["metadatas"])
meta_df["id"] = all_chunks["ids"]

print("row count:", len(meta_df))
print(meta_df["chunk_type"].value_counts())
print("unique app_ids:", meta_df["app_id"].nunique())
print()
print("null counts (review rows should have review_id/votes_helpful/recommended; description rows should not):")
print(meta_df.groupby("chunk_type")[["review_id", "votes_helpful", "recommended"]].apply(lambda g: g.notna().mean()))

row count: 16010
chunk_type
review         15695
description      315
Name: count, dtype: int64
unique app_ids: 315

null counts (review rows should have review_id/votes_helpful/recommended; description rows should not):
             review_id  votes_helpful  recommended
chunk_type                                        
description        0.0            0.0          0.0
review             1.0            1.0          1.0


In [6]:
review_rows = meta_df[meta_df["chunk_type"] == "review"]
print("votes_helpful describe:")
print(review_rows["votes_helpful"].astype(float).describe())
print()
print("recommended value counts:")
print(review_rows["recommended"].value_counts())
print()
per_game_counts = review_rows.groupby("app_id").size()
print("reviews-per-game describe (expect mostly 50, min could be lower for short-tail games):")
print(per_game_counts.describe())

votes_helpful describe:
count    15695.000000
mean       322.070468
std        854.281735
min          0.000000
25%         37.000000
50%        101.000000
75%        282.000000
max      29608.000000
Name: votes_helpful, dtype: float64

recommended value counts:
recommended
True     10506
False     5189
Name: count, dtype: int64

reviews-per-game describe (expect mostly 50, min could be lower for short-tail games):
count    315.000000
mean      49.825397
std        2.196840
min       20.000000
25%       50.000000
50%       50.000000
75%       50.000000
max       50.000000
dtype: float64


## Validate Data Quality — `game_profiles`

In [7]:
profile_meta = game_profiles.get(include=["metadatas"])
profile_df = pd.DataFrame(profile_meta["metadatas"])
profile_df["id"] = profile_meta["ids"]

print("row count:", len(profile_df))
print(profile_df["variant"].value_counts())
print()
print("expected 315 games x 4 variants - 2 skips (recommended_only, zero-eligible-review game) = 1258")

row count: 1258
variant
any_polarity__flat                315
any_polarity__log_weighted        315
recommended_only__flat            314
recommended_only__log_weighted    314
Name: count, dtype: int64

expected 315 games x 4 variants - 2 skips (recommended_only, zero-eligible-review game) = 1258


In [14]:
variant_spec = {
    "any_polarity__flat": ("any_polarity", "flat", "mean(review_vecs)"),
    "any_polarity__log_weighted": ("any_polarity", "log_weighted", "sum(w * review_vecs), w ~ log1p(votes_helpful)"),
    "recommended_only__flat": ("recommended_only", "flat", "mean(review_vecs) on recommended==1 subset"),
    "recommended_only__log_weighted": ("recommended_only", "log_weighted", "sum(w * review_vecs) on recommended==1 subset"),
}
variant_counts = profile_df["variant"].value_counts()

variant_table = pd.DataFrame(
    [
        {"variant": v, "polarity": p, "weighting": w, "pooling_formula": f, "n_rows": variant_counts.get(v, 0)}
        for v, (p, w, f) in variant_spec.items()
    ]
)
variant_table

,variant,polarity,weighting,pooling_formula,n_rows
0,any_polarity__flat,any_polarity,flat,mean(review_vecs),315
1,any_polarity__log_weighted,any_polarity,log_weighted,"sum(w * review_vecs), w ~ log1p(votes_helpful)",315
2,recommended_only__flat,recommended_only,flat,mean(review_vecs) on recommended==1 subset,314
3,recommended_only__log_weighted,recommended_only,log_weighted,sum(w * review_vecs) on recommended==1 subset,314


In [8]:
print("blend_weight unique values:", profile_df["blend_weight"].unique())
print()
print("n_reviews_pooled describe by variant:")
print(profile_df.groupby("variant")["n_reviews_pooled"].describe())
print()
print("recommended_rate describe by variant:")
print(profile_df.groupby("variant")["recommended_rate"].describe())

blend_weight unique values: [0.1]

n_reviews_pooled describe by variant:
                                count       mean        std   min    25%   50%   75%   max
variant                                                                                   
any_polarity__flat              315.0  49.825397   2.196840  20.0  50.00  50.0  50.0  50.0
any_polarity__log_weighted      315.0  49.825397   2.196840  20.0  50.00  50.0  50.0  50.0
recommended_only__flat          314.0  33.458599  13.204942   2.0  24.25  37.0  45.0  50.0
recommended_only__log_weighted  314.0  33.458599  13.204942   2.0  24.25  37.0  45.0  50.0

recommended_rate describe by variant:
                                count      mean       std  min   25%   50%  75%  max
variant                                                                             
any_polarity__flat              315.0  0.668825  0.264695  0.0  0.49  0.74  0.9  1.0
any_polarity__log_weighted      315.0  0.668825  0.264695  0.0  0.49  0.74  0.9  1.0
re

In [10]:
# Which app_ids are missing from which variant (should only be the 1 known
# recommended_only skip -- confirms Stage 2's printed warning matches the data).
all_app_ids = set(review_rows["app_id"].unique().tolist())
for variant, g in profile_df.groupby("variant"):
    missing = all_app_ids - set(g["app_id"].tolist())
    print(f"{variant}: missing app_ids = {sorted(missing)}")

any_polarity__flat: missing app_ids = []
any_polarity__log_weighted: missing app_ids = []
recommended_only__flat: missing app_ids = [285190]
recommended_only__log_weighted: missing app_ids = [285190]


## Embedding Sanity

In [11]:
sample = game_profiles.get(limit=20, include=["embeddings"])
norms = np.linalg.norm(np.asarray(sample["embeddings"]), axis=1)
print("embedding dim:", np.asarray(sample["embeddings"]).shape[1])
print("norms (should be ~1.0, L2-normalized):", norms.round(4))

embedding dim: 512
norms (should be ~1.0, L2-normalized): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## Nearest-Neighbor Spot Checks

Uses `ChromaGameProfileRetriever` directly (same class Stage 3 will register into the eval
contract) rather than raw `chromadb` calls, so this doubles as a smoke test of that class.

In [12]:
from steam_review_ml.recommender.chroma_retrieve import ChromaGameProfileRetriever

retriever = ChromaGameProfileRetriever(variant="any_polarity__flat", repo_root=REPO_ROOT)

igdb = pd.read_parquet(REPO_ROOT / "artifacts" / "igdb" / "igdb_games__enriched.parquet", columns=["app_id", "app_name"])
name_by_app_id = dict(zip(igdb["app_id"], igdb["app_name"]))

def show_neighbors(game_profiles, app_id: int, k: int = 5) -> pd.DataFrame:
    vec = game_profiles.get(ids=[f"{app_id}::any_polarity__flat"], include=["embeddings"])["embeddings"][0]
    res = game_profiles.query(
        query_embeddings=[vec],
        n_results=k,
        where={"$and": [{"variant": "any_polarity__flat"}, {"app_id": {"$ne": app_id}}]},
    )
    out = pd.DataFrame({
        "app_id": [int(m["app_id"]) for m in res["metadatas"][0]],
        "distance": res["distances"][0],
    })
    out["app_name"] = out["app_id"].map(name_by_app_id)
    return out

print("Half-Life (70) neighbors:")
display(show_neighbors(game_profiles, 70))

Half-Life (70) neighbors:


,app_id,distance,app_name
0,362890,0.099086,Black Mesa
1,546560,0.143775,Half-Life: Alyx
2,723390,0.154169,Hunt Down The Freeman
3,620,0.181533,Portal 2
4,420,0.182603,Half-Life 2: Episode Two


In [13]:
# Spot-check a second, different-genre game (excluding Half-Life so this is genuinely a different game).
other_app_ids = review_rows.loc[review_rows["app_id"] != 70, "app_id"].unique()
sample_app_id = int(other_app_ids[len(other_app_ids) // 2])
print(f"{name_by_app_id.get(sample_app_id, sample_app_id)} ({sample_app_id}) neighbors:")
display(show_neighbors(game_profiles, sample_app_id))

BERSERK and the Band of the Hawk (502280) neighbors:


,app_id,distance,app_name
0,899440,0.164022,GOD EATER 3
1,7510,0.166730,X-Blades
2,524580,0.169306,Fairy Fencer F Advent Dark Force
3,730310,0.172577,DYNASTY WARRIORS 9
4,814380,0.172735,Sekiro™: Shadows Die Twice


# Key Findings

- **Counts match:** 16,010 chunk rows (15,695 review + 315 description, 315 games); 1,258
  profile rows (315/315 `any_polarity__*`, 314/315 `recommended_only__*`). Missing `app_id`
  (285190) is the same game across both `recommended_only` variants -- matches Stage 2's printed
  skip warning.
- **Metadata sane:** `recommended_rate` = 1.0 (std 0) for `recommended_only`; spreads 0-1 (mean
  0.67) for `any_polarity`. `n_reviews_pooled` medians 50 (`any_polarity`) / 37
  (`recommended_only`). `blend_weight` uniformly 0.1.
- **Embeddings + retrieval correct:** norms exactly 1.0. Neighbors genre-appropriate for two very
  different games (Half-Life -> Black Mesa, Alyx, Portal 2; a JRPG title -> other JRPGs).

# Recommendation / Next Steps

Register `ChromaGameProfileRetriever` (`any_polarity__flat`) as a new `recs_job_eval_offline.py`
method, with Ablation B's raw-query vs. query+description arms. No data-layer risks found;
`description_blend_weight` and the other 3 variants stay deferred/available for later expansion.